SOW-BKI230A Deep Learning<br>Spring 2025

Assignment 5<br>Generative Adversarial Networks

**Name**:

Angelina Podoļako

**S-number**:

s1125886

### Generating gameboy characters with generative adversarial networks

In this assignment, you will study, complete and customise the accompanying partial implementation of a generative adversarial network by following the step-by-step instructions in the comments. Your goal is to train your generative adversarial network on the accompanying 0x72.itch.io-scraped dataset of 185472 16 × 16 pixel 2-bit gameboy-character-like images to generate the coolest characters that you can. Here is a sample of the dataset:

![](https://umguec.github.io/file-sharing/gbc_dataset_5.gif)

You should document your experiments at the end of this notebook and submit it together with a sample of your characters.

In [3]:
import random
from typing import Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from IPython.display import display
from torch.utils.data import Dataset, DataLoader

In [4]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
!wget -nc https://umguec.github.io/file-sharing/gbc_dataset.npy.zip
!unzip -n gbc_dataset.npy.zip -d assignment_5

gbc_ndarray = np.load("assignment_5/gbc_dataset.npy")

class GBCDataset(Dataset):
    """
    Implement gameboy character dataset (GBCDataset) class.

    Attributes:
        dat (np.ndarray): Data.
        dev (str): Device.
    """
    def __init__(self, data, device):
        """
        Instantiate GBCDataset class.

        Args:
            data (np.ndarray): Data.
            device (str): Device.
        """
        self.dat = data
        self.dev = device

    def __len__(self):
        """
        Get dataset cardinality.

        Returns:
            int: Dataset cardinality.
        """
        return self.dat.shape[0]

    def __getitem__(self, index):
        """
        Get dataset element.

        Args:
            index (int): Index.

        Returns:
            torch.Tensor: Dataset element.
        """
        return torch.from_numpy(self.dat[index].astype(np.float32) / 127.5 - 1.0).to(self.dev)

gbc_dataset = GBCDataset(gbc_ndarray, device)
gbc_data_loader = DataLoader(gbc_dataset, 32, True) # You can change the second argument to change the batch size.

--2025-05-14 12:42:28--  https://umguec.github.io/file-sharing/gbc_dataset.npy.zip
Resolving umguec.github.io (umguec.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to umguec.github.io (umguec.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 925054 (903K) [application/zip]
Saving to: ‘gbc_dataset.npy.zip’

gbc_dataset.npy.zip 100%[===================>] 903.37K  --.-KB/s    in 0.02s   

2025-05-14 12:42:28 (43.6 MB/s) - ‘gbc_dataset.npy.zip’ saved [925054/925054]

Archive:  gbc_dataset.npy.zip
  inflating: assignment_5/gbc_dataset.npy  
  inflating: assignment_5/__MACOSX/._gbc_dataset.npy  


In [11]:
class GBCGAN:
    """
    Implement gameboy character generative adversarial network (GBCGAN) class.

    Attributes:
        _dis_net (nn.Sequential): Discriminator network.
        _dis_opt (optim.Optimizer): Discriminator optimiser.
        _gen_net (nn.Sequential): Generator network.
        _gen_opt (optim.Optimizer): Generator optimiser.
        _lat_dim (int): Latent dimensionality.
        _los_fun (nn.Loss): Loss function.
        dev (str): Device.

    Methods:
        _get_discriminator(): Get discriminator network and discriminator optimiser.
        _get_generator(): Get generator network and generator optimiser.
        generate_images(batch_size): Generate fake images.
        train_networks(dataset, epoch_num, batch_size): Train discriminator network and generator network.
    """

    def __init__(self, device: str = "cuda") -> None:
        """
        Instantiate GBCGAN class.

        Args:
            device (str): Device.
        """
        self.dev = device
        self._dis_net, self._dis_opt = self._get_discriminator()
        self._gen_net, self._gen_opt = self._get_generator()
        self._lat_dim = self._gen_net[0].in_channels
        self._los_fun = nn.BCELoss()

    def _get_discriminator(self) -> Tuple[nn.Sequential, optim.Optimizer]:
        """
        Get discriminator network and discriminator optimiser.

        Returns:
            network (nn.Sequential): Discriminator network.
            optimiser (optim.Optimizer): Discriminator optimiser.
        """
        # QUESTION 1:
        # Instantiate the discriminator network as an nn.Sequential object whose
        #     inputs  are [batch size] × 1 × 16 × 16 fake or real iamges and
        #     outputs are [batch size] × 1 ×  1 ×  1 fake or real probabilities.
        # Move the discriminator network to the device.
        # Tip: You can try a convolutional architecture, such as:
        #     Conv2d layer (with  1 input channel , 16 output channels, kernel size 4, stride 2, padding 1, bias False), BatchNorm2d layer (with 16 output channels), LeakyReLU activation function (with negative slope 0.2)
        #     Conv2d layer (with 16 input channels, 32 output channels, kernel size 4, stride 2, padding 1, bias False), BatchNorm2d layer (with 32 output channels), LeakyReLU activation function (with negative slope 0.2)
        #     Conv2d layer (with 32 input channels,  1 output channel , kernel size 4, stride 1, padding 0, bias False), Sigmoid activation function

        # WRITE YOUR CODE BELOW:

        network = nn.Sequential(

            # The first convolutional block      =  [batch size] × 1 × 16 × 16 fake or real iamges
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1, bias=False),     #1 channel(black\white); 16=outputchannels (filters=each tries find own features), 2=2times smaller; padding=1 => zeros on edges
            nn.BatchNorm2d(16),     #normalizes activation across 16 channels (accelerates ( ускоряет) learning)
            nn.LeakyReLU(0.2, inplace=True),     #activation function with a "leak" (allows small negative values to pass through)

            # The second convolutional block     =  [batch size] × 16 × 8 × 8
            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1, bias=False),       #16=otputs from prev layer, 32=more features
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            # The third convolutional block      =  [batch size] × 32 × 4 × 4
            nn.Conv2d(32, 1, kernel_size=4, stride=1, padding=0, bias=False),   #32->1 channel (final decision: real/fake)
            nn.Sigmoid()       #converts the output to a probability (0-1);
            #final output [batch, 1, 1, 1] (one value for each image in the batch)

        ).to(self.dev)


        # QUESTION 2:
        # Instantiate the discriminator optimiser as an optim.Optimizer object.
        # Tip: You can try the Adam optimiser (with lr 0.0002, beta1 0.5, beta2 0.999).

        # WRITE YOUR CODE BELOW:
        #lr= low learning rate (GANS require careful training)
        #betas= 0.5 - momentum (smoothing gradients)      0.999 - adaptive adjustment (for rare features)
        optimiser = optim.Adam(network.parameters(), lr=0.0002, betas=(0.5, 0.999))

        return network, optimiser


    def _get_generator(self) -> Tuple[nn.Sequential, optim.Optimizer]:
        """
        Get generator network and generator optimiser.

        Returns:
            network (nn.Sequential): Generator network.
            optimiser (optim.Optimizer): Generator optimiser.
        """
        # QUESTION 3:
        # Instantiate the generator network as an nn.Sequential object whose
        #     inputs  are [batch size] × [latent dimensionality] ×  1 ×  1 random latents and
        #     outputs are [batch size] ×                       1 × 16 × 16 fake iamges.
        # Move the generator network to the device.
        # Tip: You can try a transposed convolutional architecture, such as:
        #     ConvTranspose2d layer (with 64 input channels, 32 output channels, kernel size 4, stride 1, padding 0, bias False), BatchNorm2d layer (with 32 output channels), ReLU activation function
        #     ConvTranspose2d layer (with 32 input channels, 16 output channels, kernel size 4, stride 2, padding 1, bias False), BatchNorm2d layer (with 16 output channels), ReLU activation function
        #     ConvTranspose2d layer (with 16 input channels,  1 output channel , kernel size 4, stride 2, padding 1, bias False), Tanh activation function

        #*Транспонированная свертка (иногда называемая "развертывающей сверткой") - это операция, обратная обычной свертке. Она позволяет увеличивать пространственные размеры изображения, а не уменьшать их

        # WRITE YOUR CODE BELOW:

        network = nn.Sequential(

            # The first transposed convolutional block =   [batch size] × [latent dimensionality] ×  1 ×  1 random latents  #batch= numb of generated images (eg 42);  64= dimension of latency space (size of randm vec); 1x1=initial size of image
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=1, padding=0, bias=False), #expansion to 4x4, reduced channels to 32 (output=[batch, 32, 4, 4])
            nn.BatchNorm2d(32), #normalizes activations across 32 channels
            nn.ReLU(), #activation function (all negative values become 0)

            # The second transposed convolutional block =  [batch size] × 32 × 4 × 4
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1, bias=False), #output [batch, 16, 8, 8] (less channels; bigger image size 8x8);  stride=2 in the last layers - rapid increase in size
            nn.BatchNorm2d(16),
            nn.ReLU(),

            # The third transposed convolutional block =  [batch size] × 16 × 8 × 8
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1, bias=False), #output [batch, 1,  16, 16] image size 16x16=full size; 1 channel
            nn.Tanh() #activation, which gives values in the range [-1, 1] (later converted to [0, 255])
            #[batch size] ×                       1 × 16 × 16 fake iamges

        ).to(self.dev)

        # QUESTION 4:
        # Instantiate the generator optimiser as an optim.Optimizer object.
        # Tip: You can try the Adam optimiser (with lr 0.0002, beta1 0.5, beta2 0.999).

        # WRITE YOUR CODE BELOW:

        optimiser = optim.Adam(network.parameters(), lr=0.0002, betas=(0.5, 0.999))

        return network, optimiser


    def generate_images(self, batch_size):
        """
        Generate fake images.

        Args:
            batch_size (int): Batch size.

        Returns:
            fake_images (torch.tensor): Fake images.
        """
        random_latents = torch.randn(batch_size, self._lat_dim, 1, 1, device=self.dev)
        temp_images = self._gen_net(random_latents)
        fake_images = Image.new("L", (16 * batch_size, 16))

        for i in range(batch_size):
            fake_images.paste(Image.fromarray(np.squeeze((127.5 * (temp_images[i].cpu().detach().numpy().transpose(1, 2, 0) + 1.0)).astype(np.uint8))), (16 * i, 0))

        return fake_images

    def train_networks(self, gbc_data_loader: DataLoader, epoch_number: int=1) -> None:
        """
        Train discriminator network and generator network.

        Args:
            gbc_data_loader (DataLoader): GBC data loader.
            epoch_number (int): Epochs number.
        """
        self._dis_net.train()
        self._gen_net.train()

        for epoch in range(epoch_number):
            discriminator_losses = []
            generator_losses = []

            for real_images in gbc_data_loader:
                # QUESTION 5:
                # Complete the discriminator training as follows:
                #     Sample a random latent batch (random_latents).
                #     Generate a fake image batch (fake_images).
                #     Discriminate the fake image batch (fake_probs).
                #     Define the fake label batch (fake_labels).
                #
                #     Discriminate the real image batch (real_probs).
                #     Define the real label batch (real_labels).
                #
                #     Measure the discriminator loss (discriminator_loss).

                # WRITE YOUR CODE BELOW:

                # Fake image generation
                 #Creating a random number tensor (noise) with a normal distribution (mean=0, std=1)
                random_latents = torch.randn(real_images.size(0), self._lat_dim, 1, 1, device=self.dev)    #real_images.size(0)=batch_size from input realimages ;self._lat_dim dime of latency space(64); 1x1=initial image size
                 #Passing random noise through the generator
                fake_images = self._gen_net(random_latents)   # get fake images of size [batch_size, 1, 16, 16]
                # Evaluate fake images (needs to be 0)
                fake_probs = self._dis_net(fake_images.detach())   #.detach() - detach tensor from calculation graph (so that it does not affect the training of G); Passing fake images through D; ouput= probs (0-1) that images=real
                #.detach() is critically important - without it, learning would extend to the generator.
                fake_labels = torch.zeros_like(fake_probs)   # create labels (all zeros), as these are fake images; GOAL: The discriminator must learn to return 0 for fakes.

                # Evaluate real images (needs to be 1)
                real_probs = self._dis_net(real_images)
                real_labels = torch.ones_like(real_probs)   #label=1(as real photo);  GOAL: The discriminator must learn how to return 1 for real images

                # Loss calculation
                fake_loss = self._los_fun(fake_probs, fake_labels)   #BCE: -(y*log(p) + (1-y)*log(1-p)) (where y=0 for fakes);     fine D when makes mistakes with fakes.
                real_loss = self._los_fun(real_probs, real_labels)   #same but y=1 for ral ; fine D if mistakes
                #BCELoss is well suited for binary classification (real/fake)
                discriminator_loss = discriminator_loss = (fake_loss + real_loss) / 2    #total error of D = average between the errors on fake and real images.

                self._dis_opt.zero_grad()
                discriminator_loss.backward()
                self._dis_opt.step()

                discriminator_losses.append(discriminator_loss.item())

                # QUESTION 6:
                # Complete the generator training as follows:
                #     (Re)sample a random latent batch (random_latents).
                #     (Re)generate a fake image batch (fake_images).
                #     (Re)discriminate the fake image batch (fake_probs).
                #     (Re)define the fake label batch as the real label batch (real_labels).
                #
                #     Measure the generator loss (generator_loss).

                #main goal of G= is for D to return 1 (thinking images = real)
                # WRITE YOUR CODE BELOW:

                # Generating new fake images
                random_latents = torch.randn(real_images.size(0), self._lat_dim, 1, 1, device=self.dev)   #real_images.size(0) =numb of images in current patch (same as real images);  self._lat_dim= dim of hidden space (64 in that  case)
                fake_images = self._gen_net(random_latents)   #pass noise through G -> get fake images of size [batch_size, 1, 16, 16]
                #There is no .detach() here, since we want to update the weights of the generator
                fake_probs = self._dis_net(fake_images)  #The calculation graph is saved for back propagation of the error     Passing generimages through D-> obtain probs(0-1) that D considers the images to be real
                real_labels = torch.ones_like(fake_probs)   #create "real" labels=all 1)  GOAL: We want to "trick" D into thinking fakes are real.;   Size: Same as for fake_probs (usually [batch_size, 1, 1, 1])

                # Loss calculation
                generator_loss = self._los_fun(fake_probs, real_labels)   #BCE -(y*log(p) + (1-y)*log(1-p)) (where y=1)   -> fine G when D correctly detects fakes

                self._gen_opt.zero_grad()
                generator_loss.backward()
                self._gen_opt.step()

                generator_losses.append(generator_loss.item())

            print(f"Epoch: [{epoch + 1}/{epoch_number}]")
            print(f"Discriminator loss: {np.mean(discriminator_losses):.4f}")
            print(f"Generator loss: {np.mean(generator_losses):.4f}")

        self._dis_net.eval()
        self._gen_net.eval()

gbcgan = GBCGAN(device)

*You* can use the following code cells to train your networks and generate your images:

In [12]:
gbcgan.train_networks(gbc_data_loader, 10) # You can change the argument to change the epoch number.

Epoch: [1/10]
Discriminator loss: 0.0562
Generator loss: 4.3600
Epoch: [2/10]
Discriminator loss: 0.0079
Generator loss: 6.9830
Epoch: [3/10]
Discriminator loss: 0.0068
Generator loss: 7.7731
Epoch: [4/10]
Discriminator loss: 0.0067
Generator loss: 8.1592
Epoch: [5/10]
Discriminator loss: 0.0051
Generator loss: 8.5341
Epoch: [6/10]
Discriminator loss: 0.0058
Generator loss: 8.7024
Epoch: [7/10]
Discriminator loss: 0.0053
Generator loss: 8.9733
Epoch: [8/10]
Discriminator loss: 0.0044
Generator loss: 9.2659
Epoch: [9/10]
Discriminator loss: 0.0056
Generator loss: 9.3921
Epoch: [10/10]
Discriminator loss: 0.0040
Generator loss: 9.8371


In [13]:
fake_images = gbcgan.generate_images(32) # You can change the argument to change the batch size.

display(fake_images)
fake_images.save("assignment_5/fake_images.png") # You can change the argument to change the path as long as the directory exists.

Bonus question:

Considering the characteristics of the dataset, can you think of a way to make the generative adversarial network approximately twice as fast?

WRITE YOUR ANSWER BELOW:



Most likely, this is a decrease in the dimension of latent space. That is, to reduce _lat_den from 64 to 32. This will give fewer parameters = faster learning.  

This will simplify the architecture:
Reduce the number of channels in convolutions (e.g. 32→16→8 instead of 64→32→16)

Use best GPU possible:)
Operations with float 16 instead of float32